# Chapter 16 — Combining Basic Operations

Part 6: Reductions, scans, and combined recipes.

**Ladder.**

- **Method 01 — naive two-pass scalar reference:** one row and one element at a time, using the formulas as written.
- **Method 02 — fused single-pass row kernels:** one Triton program per row for sumsq/variance, stable softmax, log-sum-exp, and LayerNorm statistics.
- **Method 03 — online/streaming and fused:** online softmax carries $(m,s)$; Welford carries $(n,\mu,M_2)$; a fused LayerNorm writes the normalized row once.

**Thread.**

- **Recurring idea:** every recipe is a reduction followed by a normalization; every stability fix is either a shift or a better merge.
- **Topic chain:** sum of squares $\to$ mean/variance $\to$ max/normalize $\to$ stable softmax $\to$ log-sum-exp $\to$ online softmax $\to$ Welford variance $\to$ LayerNorm.
- **Shape thread:** one row per program; reduce the last axis; return one scalar or one row per row.
- **Stability thread:** subtract a maximum before `exp`; keep epsilon inside `rsqrt`; merge statistics instead of subtracting large squares.

**Study tip.** Take the four-element row $x=(-1,0,1,2)$.

- Compute the mean, variance, softmax, log-sum-exp, and LayerNorm by hand.
- Redo the softmax for $x+1000$: the two rows must be identical.
- Redo it without the shift: the exponentials overflow.
- If your hand rows match the chapter, every kernel is only an addressing exercise.

**Learning objectives**

- Combine two or more reductions into a recipe and state the order of the reductions.
- Derive the stable-softmax shift and the online max/denominator update, and explain why both are exact identities.
- Derive the Welford mean and $M_2$ merge formulas and use them to avoid variance cancellation.
- Write one-program-per-row Triton kernels for sum-of-squares/variance, stable softmax, log-sum-exp, and LayerNorm statistics.
- Apply the **shape rule** (reduce the last axis; keep one scalar per row) and the **stability rule** (shift before `exp`, epsilon inside `rsqrt`).
- Choose between a two-pass kernel and an online single-pass kernel by counting bytes, not instructions.

## Chapter front matter

- **Index:** [Index](../../00_index.ipynb)
- **Previous:** [Chapter 15 — Scan Operations](../../part6_reductions/ch15_scans/15_complete.ipynb)
- **Next:** [Chapter 17 — Control Flow](../../part7_construction/ch17_control_flow/17_complete.ipynb)
- **Estimated time:** 90–120 min
- **Difficulty:** Intermediate–Advanced
- **Prerequisites:**
  - Chapter 14: `tl.sum`, `tl.max`, means, and variances along an axis.
  - Chapter 15: reductions versus scans, and loop-carried state.
  - Chapter 7: `exp`, `log`, `rsqrt`, and the meaning of an overflow.
  - No prior attention or normalization implementation is assumed.
- **Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

### Two passes versus one streaming pass

```text
two-pass softmax (max pass, then normalize pass)
pass 1: load x ──► m = max(x)
pass 2: load x ──► e = exp(x - m) ──► s = sum(e) ──► p = e / s ──► store p

online softmax (one reduction pass over chunks, then one write pass)
load chunk ──► (m_b, s_b) ──► update (m, s) ──► next chunk
load chunk ──► p = exp(x - m) / s ──► store p
```

- **Two-pass cost:** reads the row twice when the row does not fit in registers.
- **Single-pass cost:** when the whole row fits, Part III collapses the passes into one load and one store.
- **Online core:** carries `(m, s)` in registers, so the statistic pass reads the row once even when the row does not fit.

### Softmax pipeline

```text
x ──► m = max(x) ──► x - m ──► exp ──► s = sum ──► divide ──► p
                              │                    │
                              └── all exponents    └── p sums to 1
                                  are <= 0
```

- Subtracting $m$ does not change the ratio.
- It moves the largest exponent to $0$ and keeps every other exponent in $(0,1]$.

### Online max and denominator update

```text
chunk statistic:  m_b = max(chunk),  s_b = sum(exp(chunk - m_b))

m_new = max(m, m_b)
s_new = s * exp(m - m_new) + s_b * exp(m_b - m_new)
```

- Both old and new contributions are referred to $m_{\text{new}}$.
- That single rescale is the whole algorithm.

### Welford update

```text
chunk statistic:  (n_b, mu_b, M2_b)

delta  = mu_b - mu
n_new  = n + n_b
mu_new = mu + delta * n_b / n_new
M2_new = M2 + M2_b + delta^2 * n * n_b / n_new
```

- The cross term $\delta^2 n n_b/n_{\text{new}}$ is the between-chunk sum of squares.
- Dropping it is the most common Welford bug.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import math

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU)

## Part I — Mathematics

- **Recipe:** a reduction followed by a normalization, or two reductions combined into one statistic.
- **Row:** $x=(x_0,\ldots,x_{N-1})\in\mathbb R^N$; every kernel in this chapter owns one row.
- **Reductions:** collapse the last axis to $s,\mu,\sigma^2,m,\operatorname{LSE}$; normalizations return $p,\hat x$.
- **Order:** each recipe reuses the previous statistic.
  - Softmax needs the maximum.
  - Log-sum-exp needs the same shifted sum.
  - Online softmax turns the maximum and sum into loop-carried state.
  - Welford turns the mean and variance into loop-carried state.
  - LayerNorm consumes both statistics at once.

### 16.1 Sum of squares

**Connection.** Builds on Chapter 14's reductions: one `tl.sum` over an elementwise square.

- **Rule.** Square every lane, then reduce the row; the result is the squared L2 norm.
- **Formula (16.1).** $s=\sum_{j=0}^{N-1}x_j^2$, $\|x\|_2=\sqrt{s}$.
- **Example.** On $x=(-1,0,1,2)$: $s=(-1)^2+0^2+1^2+2^2=1+0+1+4=6$, $\|x\|_2=\sqrt6=2.449490$.
- **Where.** RMSNorm denominator, weight decay, and the first term of the variance identity.
- **Triton.** `ss = tl.sum(x * x, axis=0)`; keep the accumulation in `float32`, even when the input is `float16` or `bfloat16`.
- **Failure.** `float16` squares overflow when $\lvert x\rvert\gt \sqrt{65504}=255.9$ and are inexact earlier; long `float32` sums can overflow above $3.4\times10^{38}$.
- **Bridge.** One pass is enough because the square is elementwise and the sum associative; the same squared sum feeds 16.2.

### 16.2 Mean and variance

**Connection.** Builds on 16.1's sum of squares and Chapter 14's mean/variance reductions.

- **Rule.** Center the row on its mean, then average the squared deviations; the biased variance divides by $N$.
- **Formula (16.2)–(16.3).** $\mu=\frac1N\sum_jx_j$, $\sigma^2=\frac1N\sum_j(x_j-\mu)^2$; the one-pass identity is $\sigma^2=\frac1N\sum_jx_j^2-\mu^2$.
- **Example.** On $x=(-1,0,1,2)$: $\mu=2/4=0.5$, $x-\mu=(-1.5,-0.5,0.5,1.5)$, $\sum_j(x_j-\mu)^2=2.25+0.25+0.25+2.25=5$, $\sigma^2=5/4=1.25$, $\sigma=1.118034$; check $6/4-0.5^2=1.25$.
- **Where.** LayerNorm and RMSNorm, batch statistics, reward normalization, and variance penalties.
- **Triton.** `mean = tl.sum(x, axis=0) / N`; `centered = tl.where(mask, x - mean, 0.0)`; `var = tl.sum(centered * centered, axis=0) / N`; the `where` removes masked lanes without changing addresses.
- **Failure.** Cancellation: at $x=10^4+(0,1,2,3)$ in `float32` the true variance is $1.25$ but $\frac1N\sum_jx_j^2-\mu^2$ gives $0.0$. Mixing biased ($N$) and unbiased ($N-1$) variance is a silent scale error.
- **Bridge.** The mean and variance become the two statistics of LayerNorm (16.8); 16.7 makes them stable in one streaming pass.

### 16.3 Maximum and normalization

**Connection.** Builds on 16.2's row statistics and Chapter 7's `tl.max`.

- **Rule.** Divide the row by its maximum; the largest entry becomes $1$.
- **Formula (16.4).** $m=\max_{0\le j\lt N}x_j$, $y_j=x_j/m$ for $m\ne0$.
- **Example.** On $x=(-1,0,1,2)$: $m=2$, $y=(-1/2,0,1/2,1)=(-0.5,0,0.5,1)$.
- **Where.** Simplest scale-invariant normalization; first step of stable softmax; cheap bound on attention scores.
- **Triton.** `m = tl.max(x, axis=0)`, `y = x / m`; for a masked max, load with `other=-inf` so masked lanes cannot win.
- **Failure.** $m=0$ gives `inf`/`nan`; $m\lt 0$ flips every sign; an all-`-inf` row gives $m=-\infty$ and `nan`. Guard the denominator, not the output: `y = x / tl.where(m != 0.0, m, 1.0)`.
- **Bridge.** The maximum is the shift that makes softmax safe, which is 16.4.

### 16.4 Stable softmax

**Connection.** Builds on 16.3's maximum and 16.2's sum: the max becomes the shift, the sum becomes the denominator.

- **Rule.** Subtract the row maximum before exponentiating, then divide by the shifted sum.
- **Formula (16.5)–(16.6).** $p_j=\frac{e^{x_j}}{\sum_ke^{x_k}}=\frac{e^{x_j-m}}{\sum_ke^{x_k-m}}$, with $m=\max_kx_k$. Derivation: multiply numerator and denominator by $e^{-m}$:

$$
\frac{e^{x_j}}{\sum_ke^{x_k}}=\frac{e^{-m}e^{x_j}}{e^{-m}\sum_ke^{x_k}}=\frac{e^{x_j-m}}{\sum_ke^{x_k-m}}.
$$

- **Why it works.** The identity holds for every finite $m$. Choosing $m=\max_kx_k$ makes every exponent $\le0$, every exponential in $(0,1]$, the largest term exactly $1$, and the denominator in $[1,N]$, so it cannot overflow.
- **Example.** On $x=(-1,0,1,2)$, $m=2$: $e^{x-m}=(e^{-3},e^{-2},e^{-1},1)=(0.049787,0.135335,0.367879,1)$, $\sum_ke^{x_k-m}=1.553001$, $p=(0.032059,0.087144,0.236883,0.643914)$; the four probabilities sum to $1.000000$.
- **Where.** Softmax turns logits into a categorical distribution: attention, classification, and sampling.
- **Triton.** `m = tl.max(x, axis=0)`, `e = tl.exp(x - m)`, `p = e / tl.sum(e, axis=0)`; load masked lanes with `other=-inf` so $e^{x-m}=0$ there.
- **Failure.** Skipping the shift: `exp` overflows above $88.7228$ in `float32` and $11.0899$ in `float16`; then the denominator is `inf` and every probability is `inf/inf = nan`. A non-maximum shift can still overflow.
- **Bridge.** The shifted sum is exactly the log-sum-exp argument, so 16.5 keeps it in log space.

### 16.5 LogSumExp

**Connection.** Builds on 16.4's shifted denominator: keep that sum, then take its log.

- **Rule.** Return the shift plus the log of the shifted exponential sum.
- **Formula (16.7)–(16.8).** $\operatorname{LSE}(x)=\ln\sum_j e^{x_j}=m+\ln\sum_j e^{x_j-m}$, $m=\max_jx_j$; $\log p_j=x_j-\operatorname{LSE}(x)$.
- **Example.** On $x=(-1,0,1,2)$: shifted sum $1.553001$, $\ln(1.553001)=0.440190$, $\operatorname{LSE}(x)=2+0.440190=2.440190$; check $e^{2.440190}=11.475218=\sum_je^{x_j}$.
- **Where.** Log-softmax, cross-entropy, log-probability scoring, and the partition function.
- **Triton.** `m = tl.max(x, axis=0)`, `lse = m + tl.log(tl.sum(tl.exp(x - m), axis=0))`, one scalar store per row; cast half precision to `float32`.
- **Failure.** Omitting the shift, or taking `log` of a denominator that underflowed to zero. An empty or all-`-inf` row has $\operatorname{LSE}=-\infty$; define whether that is an error or a sentinel.
- **Bridge.** The same shifted sum is what online softmax maintains across chunks, which is 16.6.

### 16.6 Online softmax

**Connection.** Builds on 16.5's LSE and Chapter 15's loop-carried state: keep the max and shifted sum as chunks arrive.

- **Rule.** For each chunk compute $(m_B,s_B)$; merge into the running pair by rescaling to the new maximum.
- **Formula (16.9)–(16.10).** $m_B=\max_{j\in B}x_j$, $s_B=\sum_{j\in B}e^{x_j-m_B}$, then

$$
m_{\text{new}}=\max(m,m_B),\qquad s_{\text{new}}=s\,e^{m-m_{\text{new}}}+s_B\,e^{m_B-m_{\text{new}}}.
$$

- **Derivation.** The old contribution in unnormalized units is $s\,e^m$; referring it to $m_{\text{new}}$ multiplies by $e^{m-m_{\text{new}}}$, and similarly for $s_B$. Dividing by $e^{m_{\text{new}}}$ gives the update; it is an exact rewrite, not an approximation.
- **Example.** On $x=(-1,0,1,2)$ with chunks $(-1,0)$ and $(1,2)$: $m_1=0$, $s_1=e^{-1}+1=1.367879$; $m_2=2$, $s_2=1.367879$; $m_{\text{new}}=2$, $s_{\text{new}}=1.367879e^{-2}+1.367879=1.553001$.
- **Where.** FlashAttention, streaming rows longer than shared memory, and merging partial rows from different programs.
- **Triton.** `m_new = tl.maximum(m, tl.max(x, axis=0))`; `s = s * tl.exp(m - m_new) + tl.sum(tl.exp(x - m_new), axis=0)`; `m = m_new`. Compute `m_new` before touching `s`.
- **Failure.** Overwriting `m` first makes $e^{m-m_{\text{new}}}=1$ and drops the old chunk. $m=0$ is wrong for all-negative rows (use $-\infty$). A fully masked chunk has $m_B=-\infty$; handle it before $e^{m_B-m_{\text{new}}}$.
- **Bridge.** 16.7 applies the same chunk-merge idea to the mean and variance.

### 16.7 Welford variance

**Connection.** Builds on 16.2's variance and 16.6's chunk merge: carry a triple instead of a pair.

- **Rule.** Maintain $(n,\mu,M_2)$; merge each chunk with the parallel-axis identity; divide by the final count once.
- **Formula (16.11)–(16.13).** For one value: $n'=n+1$, $\delta=x-\mu$, $\mu'=\mu+\delta/n'$, $M_2'=M_2+\delta(x-\mu')$. For a chunk:

$$
\delta=\mu_B-\mu,\quad n'=n+n_B,\quad \mu'=\mu+\delta\frac{n_B}{n'},\quad M_2'=M_2+M_{2,B}+\delta^2\frac{n\,n_B}{n'}.
$$

- **Derivation.** Split the new sum of squares into the old points, the new points, and the mean shifts: $\sum_{i=1}^{n'}(x_i-\mu')^2=M_2+M_{2,B}+n(\mu-\mu')^2+n_B(\mu_B-\mu')^2$, where $M_2=\sum_{i\in A}(x_i-\mu)^2$ and $M_{2,B}=\sum_{i\in B}(x_i-\mu_B)^2$. The cross terms vanish because each group is centered on its own mean. From the merge, $\mu-\mu'=-\delta n_B/n'$ and $\mu_B-\mu'=\delta n/n'$, so the last two terms add to $\delta^2nn_B/n'$.
- **Example.** On $x=(-1,0,1,2)$ with chunks $(-1,0)$ and $(1,2)$: $n_1=2$, $\mu_1=-0.5$, $M_{2,1}=0.5$; $n_2=2$, $\mu_2=1.5$, $M_{2,2}=0.5$. Then $\delta=2$, $n'=4$, $\mu'=-0.5+2\cdot(2/4)=0.5$, $M_2'=0.5+0.5+2^2\cdot(2\cdot2/4)=1+4=5$, and $\sigma^2=5/4=1.25$.
- **Where.** Streaming variance and parallel/chunked variance without ever forming $\sum x^2-\mu^2$.
- **Triton.** Keep three scalars per row, reduce each chunk to $(n_B,\mu_B,M_{2,B})$, merge, and divide by the final count once.
- **Failure.** Dropping the cross term $\delta^2nn_B/n'$ keeps only the within-chunk variance. Updating $\mu$ before computing $\delta$ uses the wrong center. Forgetting to zero masked lanes inflates $n_B$ and biases $\mu_B$.
- **Bridge.** LayerNorm (16.8) needs both statistics, so the next section consumes this merge.

### 16.8 LayerNorm statistics

**Connection.** Builds on 16.2's mean/variance and 16.7's stable variance: consume both statistics per row.

- **Rule.** Compute the per-row mean and variance, normalize with `rsqrt(var + eps)`, then apply per-channel $\gamma,\beta$.
- **Formula (16.14)–(16.15).** $\mu=\frac1N\sum_jx_j$, $\sigma^2=\frac1N\sum_j(x_j-\mu)^2$, $\hat x_j=(x_j-\mu)\operatorname{rsqrt}(\sigma^2+\epsilon)$, $y_j=\gamma_j\hat x_j+\beta_j$.
- **Example.** On $x=(-1,0,1,2)$, $\epsilon=10^{-5}$: $\sigma^2+\epsilon=1.25001$, $\operatorname{rsqrt}(1.25001)=0.894424$, $\hat x=(-1.5,-0.5,0.5,1.5)\cdot0.894424=(-1.341635,-0.447212,0.447212,1.341635)$; with $\gamma=1,\beta=0$, $y=\hat x$, the mean is $0$, and the variance is $1/(1+\epsilon/\sigma^2)\approx1$.
- **Where.** Transformer blocks: per-token invariance to shift and scale, then a learned per-channel affine map.
- **Triton.** One program per row: `mean`, `centered`, `var`, `rstd = tl.rsqrt(var + eps)`, `centered * rstd * w + b`; `tl.rsqrt` is one instruction, `1 / tl.sqrt` is two.
- **Failure.** Epsilon in the wrong place: `rsqrt(var) + eps` and `1/(sqrt(var)+eps)` differ; at $\sigma^2=10^{-8}$, $\epsilon=10^{-5}$, the correct factor is $316.070$ and the wrong one is $10000.00001$ ($31.6\times$). The wrong axis gives column statistics (batch RMS), not per-token LayerNorm.
- **Bridge.** The shape rule (stats $[M]$, parameters $[N]$) and the epsilon rule are the contracts every fused normalization keeps.

**Where this leads.** Part II turns these eight recipes into scalar references; Parts III–IV turn them into row-wise, online, and fused Triton kernels. The same chain reappears in attention (online softmax) and transformer blocks (LayerNorm).

## Chapter map

- **Reading rule:** every row maps a recipe to a kernel or a demo; the stability column is the contract the kernel must keep.

| § | Recipe | Formula | Shape/stability rule | Kernel/demo |
|---|---|---|---|---|
| **16.1** | Sum of squares | $s=\sum_j x_j^2$ | one scalar per row; accumulate `float32` | `sumsq_var` |
| **16.2** | Mean and variance | $\mu=\frac1N\sum_jx_j,\ \sigma^2=\frac1N\sum_j(x_j-\mu)^2$ | reduce the last axis; avoid $\overline{x^2}-\mu^2$ | `sumsq_var` |
| **16.3** | Maximum and normalization | $m=\max_jx_j,\ y_j=x_j/m$ | guard $m=0$; max over masked lanes uses `-inf` | `sumsq_var`, `softmax` |
| **16.4** | Stable softmax | $p_j=e^{x_j-m}/\sum_ke^{x_k-m}$ | $m=\max_kx_k$; exponents $\le0$; denominator in $[1,N]$ | `softmax` |
| **16.5** | LogSumExp | $\operatorname{LSE}=m+\ln\sum_je^{x_j-m}$ | one scalar per row; shift before `exp` and `log` | `lse` |
| **16.6** | Online softmax | $m'=\max(m,m_B),\ s'=se^{m-m'}+s_Be^{m_B-m'}$ | carry $(m,s)$; compute $m'$ before rescaling $s$ | `online_softmax` |
| **16.7** | Welford variance | $(n,\mu,M_2)$ merge | keep the cross term $\delta^2nn_B/n'$ | `welford_var` |
| **16.8** | LayerNorm statistics | $\hat x=(x-\mu)\operatorname{rsqrt}(\sigma^2+\epsilon)$ | stats $[M]$, $\gamma,\beta$ $[N]$; eps inside `rsqrt` | `ln_stats`, `fused_layernorm` |
| **★** | Fused LayerNorm | $y=\gamma\hat x+\beta$ | one program per row; one read, one write | `fused_layernorm` |

## Worked numerical example

- **Spine:** $x=(-1,0,1,2)$, $N=4$, $\epsilon=10^{-5}$.
- **Sum of squares:** $s=\sum_{j=0}^{3}x_j^2=1+0+1+4=6$, $\|x\|_2=\sqrt6=2.449490$.
- **Mean:** $\sum_jx_j=2$, $\mu=2/4=0.5$, $x-\mu=(-1.5,-0.5,0.5,1.5)$.
- **Variance:** $\sum_j(x_j-\mu)^2=2.25+0.25+0.25+2.25=5$, $\sigma^2=5/4=1.25$, $\sigma=1.118034$.
- **One-pass check:** $6/4-0.5^2=1.25$; the identity agrees here because the mean is small.
- **Stable softmax:** $m=2$, $e^{x-m}=(0.049787,0.135335,0.367879,1)$, $\sum_je^{x_j-m}=1.553001$, $p=(0.032059,0.087144,0.236883,0.643914)$.
- **Log-sum-exp:** $\operatorname{LSE}(x)=2+\ln(1.553001)=2.440190$; check $e^{2.440190}=11.475218$.
- **LayerNorm:** $\operatorname{rsqrt}(1.25+10^{-5})=0.894424$, $\hat x=(-1.341635,-0.447212,0.447212,1.341635)$.
- **Shift:** $x+1000=(999,1000,1001,1002)$, $m'=1002$.

| $j$ | $x_j$ | $x_j+1000$ | $(x_j+1000)-m'$ | $e^{(x_j+1000)-m'}$ |
|---:|---:|---:|---:|---:|
| 0 | $-1$ | $999$ | $-3$ | $0.049787$ |
| 1 | $0$ | $1000$ | $-2$ | $0.135335$ |
| 2 | $1$ | $1001$ | $-1$ | $0.367879$ |
| 3 | $2$ | $1002$ | $0$ | $1$ |

- **Shift invariance:** $e^{(x_j+1000)-1002}=e^{x_j-2}$, so the denominator is again $1.553001$ and the softmax is again $p=(0.032059,0.087144,0.236883,0.643914)$.
- **Overflow without the shift:** $e^{999}=e^{1000}=e^{1001}=e^{1002}=\infty$ in `float32` because `exp` overflows above $88.7228$; the naive ratio is $\infty/\infty=\mathrm{nan}$.
- **Overflow avoided with the shift:** the largest exponent is $e^0=1$ and the denominator is finite.

| quantity | value |
|---|---|
| $s=\sum x_j^2$ | $6$ |
| $\mu$ | $0.5$ |
| $\sigma^2$ | $1.25$ |
| $\sigma$ | $1.118034$ |
| softmax | $(0.032059,0.087144,0.236883,0.643914)$ |
| $\operatorname{LSE}$ | $2.440190$ |
| LayerNorm $(\gamma=1,\beta=0)$ | $(-1.341635,-0.447212,0.447212,1.341635)$ |
| softmax$(x+1000)$ | same as softmax$(x)$ |
| naive $\exp(x+1000)$ | $(\infty,\infty,\infty,\infty)$ |

- **Verification:** the next cell checks every row with assertions only.

In [ ]:
x = torch.tensor([-1.0, 0.0, 1.0, 2.0])
eps = 1e-5

# Sum of squares, mean, variance.
assert torch.isclose((x * x).sum(), torch.tensor(6.0))
assert torch.isclose(x.mean(), torch.tensor(0.5))
centered = x - x.mean()
torch.testing.assert_close(centered, torch.tensor([-1.5, -0.5, 0.5, 1.5]))
assert torch.isclose((centered * centered).sum(), torch.tensor(5.0))
var = (centered * centered).mean()
assert torch.isclose(var, torch.tensor(1.25))
assert torch.isclose(var.sqrt(), torch.tensor(1.118034), atol=1e-6)
assert torch.isclose((x * x).mean() - x.mean() ** 2, torch.tensor(1.25))

# Stable softmax and log-sum-exp.
m = x.max()
shifted = torch.exp(x - m)
torch.testing.assert_close(
    shifted,
    torch.tensor([0.049787, 0.135335, 0.367879, 1.0]),
    atol=1e-6,
    rtol=0.0,
)
denom = shifted.sum()
assert torch.isclose(denom, torch.tensor(1.553001), atol=1e-6)
p = shifted / denom
torch.testing.assert_close(
    p,
    torch.tensor([0.032059, 0.087144, 0.236883, 0.643914]),
    atol=1e-6,
    rtol=0.0,
)
assert torch.isclose(p.sum(), torch.tensor(1.0))
lse = m + torch.log(denom)
assert torch.isclose(lse, torch.tensor(2.440190), atol=1e-6)
assert torch.isclose(torch.exp(lse), torch.exp(x).sum())

# LayerNorm.
rstd = torch.rsqrt(var + eps)
assert torch.isclose(rstd, torch.tensor(0.894424), atol=1e-6)
torch.testing.assert_close(
    centered * rstd,
    torch.tensor([-1.341635, -0.447212, 0.447212, 1.341635]),
    atol=1e-6,
    rtol=0.0,
)

# Shift invariance and the naive overflow.
assert torch.equal(torch.softmax(x, 0), torch.softmax(x + 1000.0, 0))
assert torch.isfinite(torch.softmax(x + 1000.0, 0)).all()
assert torch.isinf(torch.exp(x + 1000.0)).all()

## Part II — Method 01: the scalar reference

- **Purpose:** the correctness oracle; it visits one row and one element at a time, using the formulas as written.
- **Trade-off:** far too slow for a GPU workload, but it makes every intermediate value visible.
- **Reference table:**

| recipe | reference returns |
|---|---|
| sum of squares | $s$ |
| mean and variance | $(\mu,\sigma^2)$ |
| max normalization | $x/m$ |
| stable softmax | $p$ |
| log-sum-exp | $\operatorname{LSE}$ |
| online softmax | $(p,m,s)$ from chunks |
| Welford variance | $(\mu,\sigma^2)$ from chunk merges |
| LayerNorm | $\hat x$ |

- **Chunk rule:** the online and Welford references use chunks of two, the same merge the Triton kernels perform, reduced to Python floats.

In [ ]:
def ref_sumsq(row):
    return sum(v * v for v in row)


def ref_mean_var(row):
    n = len(row)
    mean = sum(row) / n
    var = sum((v - mean) ** 2 for v in row) / n
    return mean, var


def ref_max_norm(row):
    m = max(row)
    return [v / m for v in row]


def ref_softmax(row):
    m = max(row)
    e = [math.exp(v - m) for v in row]
    s = sum(e)
    return [v / s for v in e]


def ref_lse(row):
    m = max(row)
    return m + math.log(sum(math.exp(v - m) for v in row))


def ref_online_softmax(row, chunk=2):
    m = -math.inf
    s = 0.0
    for start in range(0, len(row), chunk):
        block = row[start:start + chunk]
        m_b = max(block)
        m_new = max(m, m_b)
        s = s * math.exp(m - m_new) + sum(math.exp(v - m_new) for v in block)
        m = m_new
    return [math.exp(v - m) / s for v in row], m, s


def ref_welford(row, chunk=2):
    count = 0.0
    mean = 0.0
    m2 = 0.0
    for start in range(0, len(row), chunk):
        block = row[start:start + chunk]
        n_b = float(len(block))
        mean_b = sum(block) / n_b
        m2_b = sum((v - mean_b) ** 2 for v in block)
        total = count + n_b
        delta = mean_b - mean
        mean = mean + delta * n_b / total
        m2 = m2 + m2_b + delta * delta * count * n_b / total
        count = total
    return mean, m2 / count


def ref_layernorm(row, eps=1e-5):
    n = len(row)
    mean = sum(row) / n
    var = sum((v - mean) ** 2 for v in row) / n
    inv = 1.0 / math.sqrt(var + eps)
    return [(v - mean) * inv for v in row]

In [ ]:
ROWS = [
    [-1.0, 0.0, 1.0, 2.0],
    [3.0, 3.0, 3.0, 3.0],
    [-2.0, -1.0, 0.0, 1.0],
]

for row in ROWS:
    t = torch.tensor(row)

    assert math.isclose(ref_sumsq(row), (t * t).sum().item(), rel_tol=1e-7)
    mean, var = ref_mean_var(row)
    assert math.isclose(mean, t.mean().item(), rel_tol=1e-7, abs_tol=1e-7)
    assert math.isclose(var, t.var(unbiased=False).item(), rel_tol=1e-6, abs_tol=1e-7)

    torch.testing.assert_close(
        torch.tensor(ref_max_norm(row)), t / t.max(), atol=1e-7, rtol=1e-7
    )
    torch.testing.assert_close(
        torch.tensor(ref_softmax(row)), torch.softmax(t, 0), atol=1e-7, rtol=1e-7
    )
    assert math.isclose(ref_lse(row), torch.logsumexp(t, 0).item(), rel_tol=1e-7)

    p_online, m_online, s_online = ref_online_softmax(row, chunk=2)
    torch.testing.assert_close(
        torch.tensor(p_online), torch.softmax(t, 0), atol=1e-7, rtol=1e-7
    )
    assert math.isclose(s_online, torch.exp(t - t.max()).sum().item(), rel_tol=1e-7)

    mean_w, var_w = ref_welford(row, chunk=2)
    assert math.isclose(mean_w, t.mean().item(), rel_tol=1e-6, abs_tol=1e-7)
    assert math.isclose(var_w, t.var(unbiased=False).item(), rel_tol=1e-6, abs_tol=1e-7)

    torch.testing.assert_close(
        torch.tensor(ref_layernorm(row)), (t - t.mean()) / torch.sqrt(t.var(unbiased=False) + 1e-5),
        atol=1e-7, rtol=1e-7,
    )

print("method 01: eight recipes match PyTorch on the spine")

## Part III — Method 02: row-wise Triton kernels

- **Launch map:** $\texttt{row}=\texttt{tl.program\_id}(0)$, $\texttt{cols}=\texttt{tl.arange}(0,\texttt{BLOCK\_N})$, $\texttt{mask}=(\texttt{row}\lt M)\land(\texttt{cols}\lt N)$.
- **Tile map:** for $M=3$, $N=6$, and $\texttt{BLOCK\_N}=8$:

| `pid` | row | cols | mask | live lanes |
|---|---|---|---|---|
| 0 | 0 | 0,1,2,3,4,5,6,7 | T,T,T,T,T,T,F,F | 0–5 |
| 1 | 1 | 0,1,2,3,4,5,6,7 | T,T,T,T,T,T,F,F | 0–5 |
| 2 | 2 | 0,1,2,3,4,5,6,7 | T,T,T,T,T,T,F,F | 0–5 |

- **Masked lanes:** the two masked lanes still execute; they must not contribute to a reduction.
- **Neutral `other`:**

| reduction | `other` | why |
|---|---|---|
| sum, sum of squares, mean | `0.0` | additive identity |
| variance after centering | `0.0` | masked lane is excluded with `tl.where` |
| max, softmax, log-sum-exp | `-inf` | cannot win the maximum and $e^{-\infty}=0$ |

- **Store shape:** two kernels store one scalar per row (sum-of-squares/variance and log-sum-exp); softmax and LayerNorm store one row per row.
- **Test timing:** the test functions are defined here and executed in the final dispatch cell, so kernel results are reported once.

In [ ]:
@triton.jit
def sumsq_var_kernel(x_ptr, sumsq_ptr, mean_ptr, var_ptr, M, N,
                     BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = (row < M) & (cols < N)

    x = tl.load(x_ptr + row * N + cols, mask=mask, other=0.0)
    mean = tl.sum(x, axis=0) / N
    centered = tl.where(mask, x - mean, 0.0)

    tl.store(sumsq_ptr + row, tl.sum(x * x, axis=0))
    tl.store(mean_ptr + row, mean)
    tl.store(var_ptr + row, tl.sum(centered * centered, axis=0) / N)


def sumsq_var(x, BLOCK_N=None):
    M, N = x.shape
    BLOCK_N = BLOCK_N or triton.next_power_of_2(N)
    ss = torch.empty(M, device=x.device, dtype=x.dtype)
    mean = torch.empty_like(ss)
    var = torch.empty_like(ss)
    sumsq_var_kernel[(M,)](x, ss, mean, var, M, N, BLOCK_N=BLOCK_N)
    return ss, mean, var

In [ ]:
def test_sumsq_var():
    torch.manual_seed(0)
    x = torch.randn(3, 7, device=DEVICE) * 4.0
    ss, mean, var = sumsq_var(x)
    assert ss.shape == mean.shape == var.shape == (3,)

    torch.testing.assert_close(ss, (x * x).sum(dim=1))
    torch.testing.assert_close(mean, x.mean(dim=1))
    torch.testing.assert_close(var, x.var(dim=1, unbiased=False))

    # A constant row has zero variance and an exact sum of squares.
    c = torch.full((2, 5), 3.0, device=DEVICE)
    ss_c, mean_c, var_c = sumsq_var(c)
    torch.testing.assert_close(ss_c, torch.full((2,), 45.0, device=DEVICE))
    torch.testing.assert_close(mean_c, torch.full((2,), 3.0, device=DEVICE))
    torch.testing.assert_close(var_c, torch.zeros(2, device=DEVICE))

    # The tile map from the text: M=3, N=6, BLOCK_N=8 -> two masked lanes.
    z = torch.randn(3, 6, device=DEVICE)
    ss_z, mean_z, var_z = sumsq_var(z, BLOCK_N=8)
    assert ss_z.shape == mean_z.shape == var_z.shape == (3,)
    torch.testing.assert_close(ss_z, (z * z).sum(dim=1))
    torch.testing.assert_close(mean_z, z.mean(dim=1))
    torch.testing.assert_close(var_z, z.var(dim=1, unbiased=False))

In [ ]:
@triton.jit
def softmax_kernel(x_ptr, out_ptr, M, N, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = (row < M) & (cols < N)

    x = tl.load(x_ptr + row * N + cols, mask=mask, other=-float("inf"))
    m = tl.max(x, axis=0)
    e = tl.exp(x - m)
    p = e / tl.sum(e, axis=0)

    tl.store(out_ptr + row * N + cols, p, mask=mask)


def softmax(x, BLOCK_N=None):
    M, N = x.shape
    BLOCK_N = BLOCK_N or triton.next_power_of_2(N)
    out = torch.empty_like(x)
    softmax_kernel[(M,)](x, out, M, N, BLOCK_N=BLOCK_N)
    return out

In [ ]:
def test_softmax():
    torch.manual_seed(0)
    x = torch.randn(4, 7, device=DEVICE) * 30.0
    out = softmax(x)
    assert out.shape == x.shape == (4, 7)
    torch.testing.assert_close(out, torch.softmax(x, dim=1))

    # Large logits: the shift is what keeps this finite.
    y = torch.tensor([[1000.0, 1001.0, 1002.0],
                      [-1000.0, 0.0, 1000.0]], device=DEVICE)
    out = softmax(y)
    assert torch.isfinite(out).all()
    torch.testing.assert_close(out, torch.softmax(y, dim=1))

    # Shift invariance of the kernel itself.
    torch.testing.assert_close(softmax(x + 100.0), softmax(x), atol=1e-6, rtol=1e-6)

In [ ]:
@triton.jit
def lse_kernel(x_ptr, out_ptr, M, N, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = (row < M) & (cols < N)

    x = tl.load(x_ptr + row * N + cols, mask=mask, other=-float("inf"))
    m = tl.max(x, axis=0)
    lse = m + tl.log(tl.sum(tl.exp(x - m), axis=0))

    tl.store(out_ptr + row, lse)


def lse(x, BLOCK_N=None):
    M, N = x.shape
    BLOCK_N = BLOCK_N or triton.next_power_of_2(N)
    out = torch.empty(M, device=x.device, dtype=x.dtype)
    lse_kernel[(M,)](x, out, M, N, BLOCK_N=BLOCK_N)
    return out

In [ ]:
def test_lse():
    torch.manual_seed(0)
    x = torch.randn(4, 7, device=DEVICE) * 4.0
    out = lse(x)
    assert out.shape == (4,)
    torch.testing.assert_close(out, torch.logsumexp(x, dim=1))

    # A large shift changes nothing but must not overflow.
    y = x + 1000.0
    torch.testing.assert_close(lse(y), torch.logsumexp(y, dim=1), atol=1e-5, rtol=1e-6)
    assert torch.isfinite(lse(y)).all()

In [ ]:
@triton.jit
def ln_stats_kernel(x_ptr, mean_ptr, rstd_ptr, M, N, eps,
                    BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = (row < M) & (cols < N)

    x = tl.load(x_ptr + row * N + cols, mask=mask, other=0.0)
    mean = tl.sum(x, axis=0) / N
    centered = tl.where(mask, x - mean, 0.0)
    var = tl.sum(centered * centered, axis=0) / N

    tl.store(mean_ptr + row, mean)
    tl.store(rstd_ptr + row, tl.rsqrt(var + eps))


def ln_stats(x, eps=1e-5, BLOCK_N=None):
    M, N = x.shape
    BLOCK_N = BLOCK_N or triton.next_power_of_2(N)
    mean = torch.empty(M, device=x.device, dtype=x.dtype)
    rstd = torch.empty_like(mean)
    ln_stats_kernel[(M,)](x, mean, rstd, M, N, eps, BLOCK_N=BLOCK_N)
    return mean, rstd

In [ ]:
def test_ln_stats():
    torch.manual_seed(0)
    x = torch.randn(4, 7, device=DEVICE) * 3.0 + 2.0
    mean, rstd = ln_stats(x, eps=1e-5)
    assert mean.shape == rstd.shape == (4,)
    torch.testing.assert_close(mean, x.mean(dim=1))
    torch.testing.assert_close(
        rstd, torch.rsqrt(x.var(dim=1, unbiased=False) + 1e-5)
    )

    # Constant row: variance is zero, so epsilon is the only guard.
    c = torch.full((2, 5), 7.0, device=DEVICE)
    mean_c, rstd_c = ln_stats(c, eps=1e-5)
    torch.testing.assert_close(mean_c, torch.full((2,), 7.0, device=DEVICE))
    torch.testing.assert_close(rstd_c, torch.full((2,), 1.0 / math.sqrt(1e-5), device=DEVICE))


print("method 02: four row-wise kernels ready")

## Part IV — Method 03: online and fused kernels

- **Why:** the Part III kernels keep the row in registers; Method 03 removes that assumption.
- **Online softmax:** carries $(m,s)$ across chunks and writes the normalized row in a second pass; the row is never materialized.
- **Welford variance:** carries $(n,\mu,M_2)$ across chunks and merges each chunk with the parallel-axis identity.
- **Fused LayerNorm:** computes the stats, normalizes, applies $\gamma$ and $\beta$, and writes the output in one program.
- **CPU check:** the CPU cells verify the chunk arithmetic itself; the kernel tests use the same chunk size and run in the final dispatch.

In [ ]:
@triton.jit
def online_softmax_kernel(x_ptr, out_ptr, N,
                          BLOCK: tl.constexpr, NUM_CHUNKS: tl.constexpr):
    row = tl.program_id(0)

    m = -float("inf")
    s = 0.0
    for k in tl.static_range(0, NUM_CHUNKS):
        cols = k * BLOCK + tl.arange(0, BLOCK)
        mask = cols < N
        x = tl.load(x_ptr + row * N + cols, mask=mask, other=-float("inf"))

        m_new = tl.maximum(m, tl.max(x, axis=0))
        s = s * tl.exp(m - m_new) + tl.sum(tl.exp(x - m_new), axis=0)
        m = m_new

    for k in tl.static_range(0, NUM_CHUNKS):
        cols = k * BLOCK + tl.arange(0, BLOCK)
        mask = cols < N
        x = tl.load(x_ptr + row * N + cols, mask=mask, other=-float("inf"))
        tl.store(out_ptr + row * N + cols, tl.exp(x - m) / s, mask=mask)


def online_softmax(x, BLOCK=4):
    M, N = x.shape
    out = torch.empty_like(x)
    online_softmax_kernel[(M,)](
        x, out, N, BLOCK=BLOCK, NUM_CHUNKS=triton.cdiv(N, BLOCK)
    )
    return out

In [ ]:
x = [-1.0, 0.0, 1.0, 2.0]
p_online, m_online, s_online = ref_online_softmax(x, chunk=2)

assert math.isclose(m_online, 2.0)
assert math.isclose(s_online, 1.553001792775919)
assert math.isclose(s_online, sum(math.exp(v - 2.0) for v in x))
torch.testing.assert_close(
    torch.tensor(p_online), torch.softmax(torch.tensor(x), 0), atol=1e-7, rtol=1e-7
)

# The same row shifted by 1000 has the same softmax and denominator.
p_shift, m_shift, s_shift = ref_online_softmax([v + 1000.0 for v in x], chunk=2)
assert math.isclose(m_shift, 1002.0)
assert math.isclose(s_shift, s_online, rel_tol=1e-12)
torch.testing.assert_close(torch.tensor(p_shift), torch.tensor(p_online), atol=1e-12, rtol=1e-12)

# An all-negative row must start from m = -inf, not m = 0.
p_neg, m_neg, s_neg = ref_online_softmax([-5.0, -4.0, -3.0], chunk=2)
assert math.isclose(m_neg, -3.0)
torch.testing.assert_close(
    torch.tensor(p_neg), torch.softmax(torch.tensor([-5.0, -4.0, -3.0]), 0),
    atol=1e-7, rtol=1e-7,
)

print("online: two-chunk merge matches the one-shot denominator")

In [ ]:
def test_online_softmax():
    torch.manual_seed(0)
    x = torch.randn(4, 7, device=DEVICE) * 4.0
    out = online_softmax(x, BLOCK=4)
    assert out.shape == x.shape == (4, 7)
    torch.testing.assert_close(out, torch.softmax(x, dim=1))

    # A chunked row with a large shift exercises the rescale.
    y = x + 500.0
    torch.testing.assert_close(
        online_softmax(y, BLOCK=4), torch.softmax(y, dim=1), atol=1e-6, rtol=1e-6
    )

In [ ]:
@triton.jit
def welford_kernel(x_ptr, mean_ptr, var_ptr, N,
                   BLOCK: tl.constexpr, NUM_CHUNKS: tl.constexpr):
    row = tl.program_id(0)

    count = 0.0
    mean = 0.0
    m2 = 0.0
    for k in tl.static_range(0, NUM_CHUNKS):
        cols = k * BLOCK + tl.arange(0, BLOCK)
        mask = cols < N
        x = tl.load(x_ptr + row * N + cols, mask=mask, other=0.0)

        n_b = tl.sum(tl.where(mask, 1.0, 0.0), axis=0)
        mean_b = tl.sum(tl.where(mask, x, 0.0), axis=0) / tl.maximum(n_b, 1.0)
        m2_b = tl.sum(tl.where(mask, (x - mean_b) * (x - mean_b), 0.0), axis=0)

        total = count + n_b
        delta = mean_b - mean
        mean = mean + delta * n_b / tl.maximum(total, 1.0)
        m2 = m2 + m2_b + delta * delta * count * n_b / tl.maximum(total, 1.0)
        count = total

    tl.store(mean_ptr + row, mean)
    tl.store(var_ptr + row, m2 / tl.maximum(count, 1.0))


def welford_var(x, BLOCK=4):
    M, N = x.shape
    mean = torch.empty(M, device=x.device, dtype=x.dtype)
    var = torch.empty_like(mean)
    welford_kernel[(M,)](
        x, mean, var, N, BLOCK=BLOCK, NUM_CHUNKS=triton.cdiv(N, BLOCK)
    )
    return mean, var

In [ ]:
x = [-1.0, 0.0, 1.0, 2.0]
mean_w, var_w = ref_welford(x, chunk=2)
assert math.isclose(mean_w, 0.5)
assert math.isclose(var_w, 1.25)

# The merge is exact for the chunk statistics shown in the text.
n1, n2 = 2.0, 2.0
mu1, mu2 = -0.5, 1.5
m2_1, m2_2 = 0.5, 0.5
delta = mu2 - mu1
total = n1 + n2
mu_merged = mu1 + delta * n2 / total
m2_merged = m2_1 + m2_2 + delta * delta * n1 * n2 / total
assert math.isclose(mu_merged, 0.5)
assert math.isclose(m2_merged, 5.0)
assert math.isclose(m2_merged / total, 1.25)

# The same merge at a large mean: the naive identity cancels in float32.
big = [10000.0, 10001.0, 10002.0, 10003.0]
mean_big, var_big = ref_welford(big, chunk=2)
assert math.isclose(mean_big, 10001.5)
assert math.isclose(var_big, 1.25)

print("welford: chunk merge keeps variance 1.25 at mean 1e4")

In [ ]:
def test_welford():
    torch.manual_seed(0)
    x = torch.randn(4, 7, device=DEVICE) * 4.0
    mean, var = welford_var(x, BLOCK=4)
    assert mean.shape == var.shape == (4,)
    torch.testing.assert_close(mean, x.mean(dim=1), atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(var, x.var(dim=1, unbiased=False), atol=1e-5, rtol=1e-5)

    # Large mean, small variance: the two-pass form is still accurate.
    y = 1e4 + torch.arange(28, device=DEVICE, dtype=torch.float32).reshape(4, 7)
    mean_y, var_y = welford_var(y, BLOCK=4)
    torch.testing.assert_close(mean_y, y.mean(dim=1), atol=1e-2, rtol=1e-7)
    torch.testing.assert_close(var_y, y.var(dim=1, unbiased=False), atol=1e-3, rtol=1e-4)

In [ ]:
@triton.jit
def fused_layernorm_kernel(x_ptr, w_ptr, b_ptr, out_ptr, M, N, eps,
                           BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = (row < M) & (cols < N)

    x = tl.load(x_ptr + row * N + cols, mask=mask, other=0.0)
    mean = tl.sum(x, axis=0) / N
    centered = tl.where(mask, x - mean, 0.0)
    var = tl.sum(centered * centered, axis=0) / N
    rstd = tl.rsqrt(var + eps)

    w = tl.load(w_ptr + cols, mask=cols < N, other=1.0)
    b = tl.load(b_ptr + cols, mask=cols < N, other=0.0)
    tl.store(out_ptr + row * N + cols, centered * rstd * w + b, mask=mask)


def fused_layernorm(x, w, b, eps=1e-5, BLOCK_N=None):
    M, N = x.shape
    BLOCK_N = BLOCK_N or triton.next_power_of_2(N)
    out = torch.empty_like(x)
    fused_layernorm_kernel[(M,)](x, w, b, out, M, N, eps, BLOCK_N=BLOCK_N)
    return out

In [ ]:
def test_fused_layernorm():
    torch.manual_seed(0)
    M, N = 4, 7
    x = torch.randn(M, N, device=DEVICE) * 3.0 + 2.0
    w = torch.randn(N, device=DEVICE)
    b = torch.randn(N, device=DEVICE)

    expected = torch.nn.functional.layer_norm(x, (N,), w, b, 1e-5)
    out = fused_layernorm(x, w, b)
    assert out.shape == x.shape == (M, N)
    torch.testing.assert_close(out, expected, atol=1e-5, rtol=1e-5)


print("method 03: online and fused kernels ready")

## Numerical stability

- **Stability is not separate:** it is the set of rules that makes the recipes correct on the finite exponent range of the hardware.

### Overflow and underflow thresholds

- **`float32`:** largest finite value $3.4028235\times10^{38}$, so `exp` overflows at $x\gt \ln(3.4028235\times10^{38})=88.7228$ (16.17).
- **`float16`:** largest finite value $65504$, so `exp` overflows at $x\gt \ln(65504)=11.0899$ (16.18).
- **Underflow:** to zero near $\ln(1.4\times10^{-45})=-103.28$ in `float32`.
- **Shift guarantee:** every exponent is $\le0$; values below $-103.28$ underflow to $0$, the correct limit.

### Shift invariance

- **Identity:** $\frac{e^{x_j-m}}{\sum_ke^{x_k-m}}=\frac{e^{x_j}}{\sum_ke^{x_k}}$ for any finite $m$ (16.19).
- **Consequence:** softmax is invariant under $x\mapsto x+c$ for any constant $c$.
- **Spine:** $c=1000$ gives the same $p$ to the last bit because the shifted values are integers and $1002-1000=2$ is exact.
- **Caveat:** if $c$ is not representable, the shift is still exact mathematically, but the input has already lost precision.

### Epsilon placement

- **Correct form:** $\operatorname{rsqrt}(\sigma^2+\epsilon)$, not $\operatorname{rsqrt}(\sigma^2)+\epsilon$ (16.20).
- **At $\sigma^2=10^{-8}$, $\epsilon=10^{-5}$:** correct $\operatorname{rsqrt}(1.00001\times10^{-5})=316.070$; wrong $\operatorname{rsqrt}(10^{-8})+10^{-5}=10000.00001$ (16.21).
- **Scale error:** the wrong placement changes the scale by $31.6\times$.
- **Placement:** epsilon belongs inside the square root, added to the variance, not the standard deviation.

### Variance cancellation

- **Identity:** $\sigma^2=\overline{x^2}-\mu^2$ loses digits when $\mu^2\gg\sigma^2$.
- **`float32` example:** $x=10^4+(0,1,2,3)$ gives $\overline{x^2}-\mu^2=0.0$ while $\sigma^2=1.25$ (16.22), a relative error of $100\%$.
- **Fix:** Welford's merge never forms the large square and returns $1.25$ at the same mean.

### Two-pass versus one-pass

- **One full pass:** for $M=1024$, $N=4096$, `float32`, $MNb=1024\cdot4096\cdot4=16\ \text{MiB}$ (16.23).

| schedule | reads | writes | traffic |
|---|---:|---:|---:|
| one-pass sum of squares | 1 | 1 | $32$ MiB |
| two-pass mean/variance | 2 | 1 | $48$ MiB |
| three-pass stable softmax | 3 | 1 | $64$ MiB |
| online reduction + write | 1 | 1 | $32$ MiB |

- **Online win:** halves the traffic of the three-pass softmax.
- **Online cost:** loop-carried scalars and a second pass that must reread the row.
- **Two-pass win:** more parallelism and no cross-chunk dependency; choose it for short rows when the reduction is not the bottleneck.

In [ ]:
x = torch.tensor([-1.0, 0.0, 1.0, 2.0])

# Shift invariance is exact for this row.
assert torch.equal(torch.softmax(x, 0), torch.softmax(x + 1000.0, 0))

# Naive exp overflows; the stable path does not.
assert torch.isinf(torch.exp(x + 1000.0)).all()
assert torch.isfinite(torch.softmax(x + 1000.0, 0)).all()

# Overflow thresholds.
thr32 = math.log(torch.finfo(torch.float32).max)
thr16 = math.log(65504.0)
assert 88.72 < thr32 < 88.73
assert 11.08 < thr16 < 11.10
assert torch.exp(torch.tensor([-104.0])).item() == 0.0

# Epsilon placement.
var_small = torch.tensor(1e-8)
proper = torch.rsqrt(var_small + 1e-5)
wrong = torch.rsqrt(var_small) + 1e-5
assert math.isclose(proper.item(), 316.06977, rel_tol=1e-5)
assert wrong.item() > 9000.0

# Variance cancellation in float32.
y = 1e4 + torch.arange(4, dtype=torch.float32)
naive_var = (y * y).mean() - y.mean() ** 2
true_var = y.var(unbiased=False)
assert math.isclose(true_var.item(), 1.25)
assert naive_var.item() == 0.0

# Traffic: one full pass is 16 MiB for this shape.
one_pass = 1024 * 4096 * 4
assert one_pass // 1024 ** 2 == 16
assert (2 * one_pass + one_pass) // 1024 ** 2 == 48
assert (3 * one_pass + one_pass) // 1024 ** 2 == 64

print("stability: thresholds, shift, epsilon, and cancellation checked")

## Pitfalls

- **Use this table as a checklist:** each row is a contract, not a performance hint.

| Pitfall | What goes wrong | Fix |
|---|---|---|
| **Missing max shift** | `exp` overflows above $88.72$ (`float32`) or $11.09$ (`float16`); the ratio becomes `inf/inf = nan` | subtract the row max (or the running max) before every `exp` |
| **Epsilon outside `rsqrt`** | `rsqrt(var) + eps` applies a different function; at $10^{-8}$ the error is $31.6\times$ | `tl.rsqrt(var + eps)`, with `eps` added to the variance |
| **Variance cancellation** | $\overline{x^2}-\mu^2$ gives $0$ instead of $1.25$ at mean $10^4$ in `float32` | use the two-pass centered form or Welford's $M_2$ |
| **Wrong axis** | reducing columns gives one statistic per column (batch RMS) instead of one per row | one program per row; `tl.sum(x, axis=0)` on the row tile |
| **Masked lanes** | zeros enter the mean/variance and bias them; a masked max can win if `other=0` | `other=0` for sums, `other=-inf` for max/softmax; `tl.where(mask, ...)` before reductions |
| **Online update order** | overwriting $m$ before rescaling $s$ drops the old chunk | compute $m_{\text{new}}$, then `s = s*exp(m-m_new) + ...`, then `m = m_new` |
| **Welford $M_2$** | omitting $\delta^2 n n_B/n'$ keeps only the within-chunk variance | merge the full triple $(n,\mu,M_2)$ with the Welford merge |

- **Zero denominator:** an empty or all-`-inf` row needs an explicit contract.
- **Biased versus unbiased:** divide by $N$ for LayerNorm, by $N-1$ for a sample estimate.

## Self-check

1. On $x=(-1,0,1,2)$, recompute $\mu$, $\sigma^2$, softmax, $\operatorname{LSE}$, and LayerNorm without looking at the worked example.
2. Prove that softmax is invariant under $x\mapsto x+c$ for any finite $c$.
3. Derive the online update from the two chunk statistics $(m_B,s_B)$ and $(m,s)$.
4. Derive the Welford merge and identify the cross term.
5. Why is $m=-\infty$ the correct initial running maximum, while $m=0$ is wrong?
6. At $\sigma^2=10^{-8}$ and $\epsilon=10^{-5}$, compare $\operatorname{rsqrt}(\sigma^2+\epsilon)$ with $\operatorname{rsqrt}(\sigma^2)+\epsilon$.
7. For $M=1024$, $N=4096$, `float32`, count the MiB moved by a three-pass softmax and by an online softmax.
8. Which reduction uses `other=-inf`, and which uses `other=0`? Give one counterexample for each wrong choice.

## Summary

- **Recipe:** a reduction plus a normalization; the reduction fixes the shape: one scalar per row, or one row per row.
- **Stable softmax:** subtract the row maximum; the shift is exact and keeps every exponent in $(0,1]$.
- **LogSumExp:** the same shifted sum in log space; it is the normalizer of the log-softmax.
- **Online softmax:** carries $(m,s)$ and rescales the old denominator by $e^{m-m_{\text{new}}}$ when a larger maximum arrives.
- **Welford:** carries $(n,\mu,M_2)$ and merges chunks with the between-chunk term $\delta^2 n n_B/n'$; it avoids the cancellation of $\overline{x^2}-\mu^2$.
- **LayerNorm:** per-row mean and variance over the last axis, then `rsqrt(var + eps)`, then a per-channel affine map.
- **Schedules:** the two-pass and one-pass forms trade traffic for dependency; the online schedule halves the reads of a three-pass softmax but carries loop state.

**Where this leads.** The exercises below revisit each contract; Chapter 17 moves from row-wise recipes to control flow, where the same kernels gain data-dependent loops.

## Exercises

1. Repeat the worked example on $x=(-2,-1,0,1)$: sumsq, mean, variance, softmax, log-sum-exp, and LayerNorm.
2. Prove the softmax shift identity for an arbitrary finite $m$, then explain why $m=\max_kx_k$ is the only choice that guarantees no overflow.
3. Merge three Welford chunks in two different orders and show that the result is the same up to floating-point rounding.
4. Replace the two-pass variance in `sumsq_var` with $\overline{x^2}-\mu^2$ and measure the error at mean $10^4$ in `float32`.
5. Implement `log_softmax(x) = x - lse(x)` and check it against `torch.log_softmax`.
6. Change `BLOCK_N` from `next_pow2(N)` to the next smaller power of two and explain what breaks. (The kernel's tile must cover the row.)
7. Add a temperature $T$: $p_j\propto e^{x_j/T}$. Which parts of the kernel change, and what happens to the shift?
8. Fuse a scale $s$ into the softmax output: $y=s\,p$. Count the saved traffic compared with two kernels.
9. For a constant row, find the smallest $\epsilon$ that keeps `rsqrt(var + eps)` finite in `float32`. Why is the output still zero?
10. For $M=2048$, $N=8192$, `float32`, compute the MiB moved by the two-pass and online schedules, and the ratio.

In [ ]:
print("CPU: numerical, reference, and stability cells already ran.")
print("Kernel tests (run on a GPU or under TRITON_INTERPRET=1):")

for fn in (
    test_sumsq_var,
    test_softmax,
    test_lse,
    test_ln_stats,
    test_online_softmax,
    test_welford,
    test_fused_layernorm,
):
    run_if_gpu(fn)

print("kernel dispatch complete")